In [ ]:
from PIL import Image
import subprocess
import os

def compress_image(input_path, filesize_threshold=500):
    """Function to compress an image using pillow and optionally ffmpeg, 
    to a maximum of the specified filesize threshold [KB]. It will first 
    convert to .jpg format and compress in multiple stages to meet the 
    file size threshold if necessary."""
    
    # Convert all non-jpg images in the input path to .jpg images 
    for filename in os.listdir(input_path):
        original_file_path = os.path.join(input_path, filename)

        # Initialize current file path
        current_file_path = original_file_path

        # Skip non-image files
        if not current_file_path.lower().endswith(('.png', '.jpg', '.jpeg', '.webp')):
            continue

        # Then convert all non-jpg images to .jpg 
        if not current_file_path.lower().endswith('jpg'):
            basename = os.path.splitext(filename)[0]
            new_file_path = os.path.join(input_path, basename+'.jpg')

            result = subprocess.run(["ffmpeg",
                            "-i", 
                            current_file_path, 
                            '-preset', 
                            'medium', 
                            new_file_path, 
                            '-y'], 
                            capture_output=True, text=True)
            
            if result.returncode != 0:
                print(f"FFmpeg failed with return code: {result.returncode}")
                print("STDOUT:", result.stdout)
                print("STDERR:", result.stderr)

            # now delete original non-jpg image
            os.remove(original_file_path)
    
    # Compress all image files in the input path that are too large
    for filename in os.listdir(input_path):
        current_file_path = os.path.join(input_path, filename)    
        
        # Skip non-image files
        if not current_file_path.lower().endswith('.jpg'):
            continue

        # First resize the image based on specified width (800px)
        with Image.open(current_file_path) as my_image: # open image with pillow

            # original image height and width
            original_image_height = my_image.height
            original_image_width = my_image.width

            print(f"Original image size: height={original_image_height}, width={original_image_width}")

            # resize the image
            new_image_width = 800 # Blog width [px]
            my_image = my_image.resize((new_image_width,int(my_image.height * (new_image_width / original_image_width))))

            new_image_height = my_image.height

            # Save resized image and overwrite original
            my_image.save(current_file_path)

            # Check new filesize after resizing
            current_filesize = os.path.getsize(current_file_path) / 1024
            print(f"Resized image dimensions: height={new_image_height}, width={new_image_width}")
            print("The filesize of resized image is: ", round(current_filesize), "KB")
        
        # Skip files that dont need compression
        if current_filesize <= filesize_threshold:
            print(f"✓ Size of {filename} already under threshold, skipping {filename}")
            continue
        else:
            print(f"Compressing {filename} further with ffmpeg...")

            compressed_file_path = current_file_path.replace('.jpg', '_temp.jpg')

            # configuring ffmpeg encoding settings
            quality_range = range(1, 31, 1)
            max_compression_value = 31
            encoder = "-q:v"

            for quality_value in quality_range:

                ffmpeg_command = [
                    "ffmpeg", 
                    "-i",
                    current_file_path,
                    "-y", 
                    encoder,
                    str(quality_value),
                ]
                ffmpeg_command.append(compressed_file_path)
                print(ffmpeg_command)

                # run ffmpeg using subprocess
                print(f"Trying quality value: {quality_value} ...")
                result = subprocess.run(ffmpeg_command, capture_output=True, text=True)
                if result.returncode != 0:
                    print(f"FFmpeg failed with return code: {result.returncode}")
                    print("STDOUT:", result.stdout)
                    print("STDERR:", result.stderr)

                # Check current filesize after compression
                if os.path.exists(compressed_file_path):
                    current_filesize = os.path.getsize(current_file_path) / 1024
                    print(f"Quality_value: {quality_value}, filesize: {round(current_filesize)} KB")
            
                    if 0 < current_filesize <= filesize_threshold:
                        print(f"Target reached with quality_value {quality_value}")
                        # # Delete original
                        # os.remove(current_file_path)
                        # # Rename temp to original name
                        # os.rename(compressed_file_path, current_file_path)
                        print(f"✓ {filename} succesfully compressed")  
                        break
                    elif current_filesize > filesize_threshold and quality_value == max_compression_value:
                        raise ValueError("Output filesize still too big after maximum compression!")
                    elif current_filesize == 0:
                        raise ValueError("Output filesize is zero KB!")
                    else:
                        raise ValueError(f"Filesize: {round(current_filesize)} KB with quality value: {quality_value}")
                
    print(f"All images in {input_path} processed!")
                
if __name__ == "__main__":

    compress_image(input_path=r"D:\Code\Project_Van_Hool\Media\webasto")

Original image size: height=794, width=850
Resized image dimensions: height=747, width=800
The filesize of resized image is:  66 KB
✓ Size of webasto1.jpg already under threshold, skipping webasto1.jpg
Original image size: height=919, width=459
Resized image dimensions: height=1601, width=800
The filesize of resized image is:  154 KB
✓ Size of webasto10.jpg already under threshold, skipping webasto10.jpg
Original image size: height=3168, width=1584
Resized image dimensions: height=1600, width=800
The filesize of resized image is:  158 KB
✓ Size of webasto11.1.jpg already under threshold, skipping webasto11.1.jpg
Original image size: height=1584, width=3168
Resized image dimensions: height=400, width=800
The filesize of resized image is:  56 KB
✓ Size of webasto11.2.jpg already under threshold, skipping webasto11.2.jpg
Original image size: height=919, width=459
Resized image dimensions: height=1601, width=800
The filesize of resized image is:  137 KB
✓ Size of webasto11.jpg already unde